# Parallelism with Threads

---

## Parallelism

Programs and algorithms can be decomposed into a sequence of tasks. We can multitask and do many computations simultaneously to achieve:
- Task-parellelism, where we execute different tasks at the same time
- Data-parallelism, where we divide some data and operate on the chunks simultaneously

Implicit parallelism (automated parallelism performed by the compiler) is nice, but very limited. We often have to deal with explicit parallelism, which is task decomposition performed manually by the programmer.

---

## Dependency

Some tasks depend on the completion of other tasks before it can commence. So if $A$ must be completed before $B$, then $A \rightarrow B$. The less dependencies that there are between tasks, the more task-parallelism there may be.

We use a directed acyclic graph to represent tasks and their dependencies.

<center>

![dependencies graph](img/dependencies.svg)

</center>

Two tasks $A$ and $B$ can execute in parallel, if and only if:
1. There is no path from $A$ to $B$
2. There is no path from $B$ to $A$

Also, dependecy is transitive. If $A \rightarrow B$ and $B \rightarrow C$, then $A \rightarrow C$.

---

## Parallelism vs. Concurrency

Both parallelism and concurrency are for multitasking, but go about it in a different way. 

Concurrency is switching between different processes. A concurrent execution of two processes involves interleaving their instructions for a single core.

Parallelism is running different tasks at the same time. A processor with multiple threads can run multiple tasks in parallel.

---

## Threads

A thread is simply a series of instructions, occurring as a single line of execution. Most programs we write are single-threaded.

They have their own:
- Thread ID
- Program Counter
- Stack
- Registers

A thread shares the virtual memory of a process with all other threads.

---

## Threads vs. Processes

Multiple threads lets us multitask in the same way that processes do. Usually, spawning new processes is overkill for parallelism purposes, and IPC requires a lot of setup. Threads, on the otherhand, are more lightweight; creating a new thread is a lot cheaper than forking.

We cannot, however, call operations such as `execl` or `sleep` in a thread, as this will affect the entire process in which the thread resides. `fork` also copies the whole process.

With the exception of the `main` thread which can terminate the whole process, all threads are considered to have equal priorities. Rather than the parent-child hierarchy found in processes, threads follow a peer relationship.

While coordination between threads is a lot easier, we can encounter more problems with race conditions. But, we also get a lot of tools to control these problems.

---

## POSIX Threads

Programs that use POSIX threads must include `<pthread.h>` and be compiled with the `-lpthread` flag.

This library includes:
- `pthread_t` which represents the "thread object"
- `pthread_create` creates and lauches a thread
- `pthread_join` blocks until the given thread has reached the end of execution

### Create

To create a thread, use the `pthread_create` function.
```c
int pthread_create(pthread_t* thread, const pthread_attr_t* attr, void* (*start_routine) (void *), void* arg);
```
It takes arguments:
1. A `pthread_t`, which it will use to store the thread ID.
2. A pointer to a `pthread_attr_t` structure, specifies additional options. Fine to leave as `NULL`.
3. A pointer to a function that the thread will execute once it spawns.
4. The `void*` parameter passed into the thread function.

It returns zero if a thread has been successfully created.

### Join

The `pthread_join` function forces the calling thread to wait for a particular thread ID to finish.
```c
int pthread_join(pthread_t thread, void** retval);
```
The first argument is the thread to wait for, the second allows you to get the return value of the original routine. If you do not need it, just pass in `NULL`. Returns zero on success.

Note that `pthread_join` is a blocking function.

---

## Terminating Threads

A thread can be terminated via calling `pthread_exit`.
```c
void pthread_exit(void* retval);
```
A thread can also be cancelled by another thread. In either case, the thread is destroyed and their resources become unavailable. Warning: this also includes unreleased dynamic memory and opened files.

---

## Critical Sections

A critical section is a set of instructions that is susceptible to race conditions.

For instance, if there exists a variable defined in the global space that multiple threads can access, then the critical section would be any of the corresponding lines of code that make use of or change that variable.

Critical sections must be taken seriously as they can cause vulnerabilities. 